# DeepWell - Observational A/B Testing & Hypothesis Testing
### Notebook 04: Pre-Validasi Sistem Rekomendasi
---

### Tujuan Notebook Ini
Tugas utama di fase ini adalah melakukan **Pre-Validation** terhadap logika bisnis sistem rekomendasi aplikasi DeepWell. Sebelum AI menyarankan pengguna untuk mengubah gaya hidup mereka (seperti menambah durasi tidur atau berolahraga), kita harus membuktikan secara statistik (menggunakan data historis) bahwa kebiasaan tersebut memang berdampak signifikan pada kesejahteraan mental dan fisik.

**Skenario Uji Hipotesis (A/B Testing Observasional):**
1. **Kasus Mental Health:** Apakah remaja dengan aktivitas fisik memadai memiliki tingkat stres yang secara signifikan lebih rendah dibanding yang jarang beraktivitas?
2. **Kasus Sleep Health:** Apakah individu dengan durasi tidur ideal (> 7 jam) memiliki tingkat stres yang secara signifikan lebih rendah dibanding yang kurang tidur (≤ 7 jam)?

*(Catatan: Pengujian ini murni menggunakan dataset bersih dari tahap Data Wrangling untuk menjaga keaslian observasi dunia nyata).*

In [4]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
sns.set_theme(style='whitegrid', palette='Set2')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup Path
BASE_PATH = "/content/drive/MyDrive/DeepWell/"
PROC_PATH = os.path.join(BASE_PATH, "data/processed/")

# Load Dataset
try:
    df_mental = pd.read_csv(os.path.join(PROC_PATH, 'teen_mental_health_clean.csv'))
    df_sleep = pd.read_csv(os.path.join(PROC_PATH, 'sleep_health_clean.csv'))

    print("[SUCCESS] Datasets loaded successfully from Google Drive.")
    print(f"Shape Mental Health : {df_mental.shape}")
    print(f"Shape Sleep Health  : {df_sleep.shape}")
except FileNotFoundError as e:
    print(f"[ERROR] Data tidak ditemukan. Cek kembali nama file atau path drive kamu: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[SUCCESS] Datasets loaded successfully from Google Drive.
Shape Mental Health : (1200, 13)
Shape Sleep Health  : (1500, 13)


## Uji 1: Dampak Aktivitas Fisik Terhadap Tingkat Stres (Mental Health)
---
* **Grup A (Treatment/Aktif):** Remaja dengan skor aktivitas fisik yang memadai (`physical_activity` ≥ 1.0).
* **Grup B (Control/Kurang):** Remaja dengan skor aktivitas fisik rendah (`physical_activity` < 1.0).
* **Metrik Evaluasi:** `stress_level`.

In [6]:
print("A/B Testing: Physical Activity vs Stress Level\n")

# Definisikan ambang batas aktivitas
threshold_activity = 1.0

# Definisikan Grup
group_a_mental = df_mental[df_mental['physical_activity'] >= threshold_activity]['stress_level']
group_b_mental = df_mental[df_mental['physical_activity'] < threshold_activity]['stress_level']

print(f"Grup A (Aktif Olahraga)  : N={len(group_a_mental)}, Rata-rata Stres={group_a_mental.mean():.2f}")
print(f"Grup B (Jarang Olahraga) : N={len(group_b_mental)}, Rata-rata Stres={group_b_mental.mean():.2f}")

# Uji Mann-Whitney U (Satu Arah: Less)
# Kita menguji apakah stres Grup A < Grup B
stat_1, p_value_1 = stats.mannwhitneyu(group_a_mental, group_b_mental, alternative='less')

print(f"\nP-Value: {p_value_1:.5f}")

if p_value_1 < 0.05:
    print("Kesimpulan: H0 DITOLAK. Grup A memiliki tingkat stres yang SECARA SIGNIFIKAN LEBIH RENDAH daripada Grup B.")
else:
    print("Kesimpulan: GAGAL MENOLAK H0. Tidak ada perbedaan signifikan.")

A/B Testing: Physical Activity vs Stress Level

Grup A (Aktif Olahraga)  : N=653, Rata-rata Stres=5.38
Grup B (Jarang Olahraga) : N=547, Rata-rata Stres=5.52

P-Value: 0.21981
Kesimpulan: GAGAL MENOLAK H0. Tidak ada perbedaan signifikan.


## Uji 2: Dampak Durasi Tidur Terhadap Stres (Sleep Health)
---
* **Grup A (Treatment/Ideal):** Individu dengan jam tidur sehat (`Sleep Duration` > 7.0 jam).
* **Grup B (Control/Kurang):** Individu dengan kurang tidur (`Sleep Duration` ≤ 7.0 jam).
* **Metrik Evaluasi:** `Stress Level`.

In [7]:
print("A/B Testing: Sleep Duration vs Stress Level\n")

# Definisikan Grup
group_a_sleep = df_sleep[df_sleep['Sleep Duration'] > 7.0]['Stress Level']
group_b_sleep = df_sleep[df_sleep['Sleep Duration'] <= 7.0]['Stress Level']

print(f"Grup A (> 7 jam tidur) : N={len(group_a_sleep)}, Rata-rata Stres={group_a_sleep.mean():.2f}")
print(f"Grup B (≤ 7 jam tidur) : N={len(group_b_sleep)}, Rata-rata Stres={group_b_sleep.mean():.2f}")

# Uji Mann-Whitney U (Satu Arah: Less)
# Menguji apakah stres Grup A < Grup B
stat_2, p_value_2 = stats.mannwhitneyu(group_a_sleep, group_b_sleep, alternative='less')

print(f"\nP-Value: {p_value_2:.10f}")

if p_value_2 < 0.05:
    print("Kesimpulan: H0 DITOLAK. Grup A memiliki tingkat stres yang SECARA SIGNIFIKAN LEBIH RENDAH daripada Grup B.")
else:
    print("Kesimpulan: GAGAL MENOLAK H0.")

A/B Testing: Sleep Duration vs Stress Level

Grup A (> 7 jam tidur) : N=1177, Rata-rata Stres=5.59
Grup B (≤ 7 jam tidur) : N=323, Rata-rata Stres=7.56

P-Value: 0.0000000000
Kesimpulan: H0 DITOLAK. Grup A memiliki tingkat stres yang SECARA SIGNIFIKAN LEBIH RENDAH daripada Grup B.


## Kesimpulan Akhir & Jembatan Menuju Evaluasi Post-Deployment (RQ3)
---
Berdasarkan uji hipotesis statistik observasional di atas, kita mendapatkan *insight* data yang sangat krusial untuk membentuk *business logic* aplikasi DeepWell:

1. **Aktivitas Fisik BUKAN Penentu Utama Penurunan Stres:** Hasil Uji 1 (P-Value 0.219) menunjukkan tidak ada perbedaan signifikan tingkat stres antara pengguna yang aktif berolahraga dengan yang jarang. Temuan empiris ini selaras dengan hasil *Exploratory Data Analysis* (EDA), yang mengindikasikan bahwa tingginya aktivitas fisik terkadang berfungsi sebagai *coping mechanism* saat seseorang mengalami stres, bukan faktor tunggal penurun stres.
2. **Tidur Ideal adalah Kunci Utama (*Game-Changer*):** Hasil Uji 2 (P-Value 0.0000) membuktikan secara absolut bahwa durasi tidur di atas 7 jam secara signifikan menurunkan tingkat stres (dari rata-rata stres 7.56 turun tajam menjadi 5.59).

---
**Implikasi Terhadap Sistem Rekomendasi AI & RQ 3:**
Pengujian *Pre-Validation* ini memberi landasan ilmiah yang tegas bagi arsitektur AI DeepWell: **Sistem rekomendasi harus memprioritaskan perbaikan pola tidur (*Sleep Health*) sebagai intervensi preventif paling utama untuk meredam stres pengguna.** Rekomendasi peningkatan aktivitas fisik tetap dapat diberikan, namun AI tidak boleh mengandalkannya sebagai solusi tunggal untuk target *stress reduction*.

Untuk menjawab RQ 3 secara utuh (*"Apakah sistem rekomendasi dapat menurunkan stres pengguna dalam siklus mingguan?"*), pengujian A/B Testing berskala **Live/Eksperimental** akan dieksekusi pada fase **Post-Deployment (Beta Testing)**, di mana *impact* nyata dari rekomendasi pola tidur oleh AI ini akan diukur secara longitudinal (*time-series*) terhadap pengguna asli selama 7 hari penuh.